# COMP3710 Lab 2 — Part 3.1: CNN Classifier

This notebook trains a simple PyTorch CNN to classify faces in the Labeled Faces in the Wild (LFW) dataset. It deliberately uses a small, explainable architecture and raw 2D images rather than flattened PCA features.

## 1. Imports

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_lfw_people
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

## 2. Random seed and device

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Load LFWUse `lfw_people.images`, not the already-flattened `lfw_people.data`. The cell below supports two environments: it uses the project cache when `data/lfw_home` exists, and otherwise lets scikit-learn download the dataset (for example, in Google Colab).

In [ ]:
# Version 1: use the existing project copy when it is available.
local_data_home = Path.cwd() / "data"
local_lfw_cache = local_data_home / "lfw_home"

if local_lfw_cache.exists():
    print(f"Loading LFW from local cache: {local_lfw_cache}")
    lfw_people = fetch_lfw_people(
        data_home=local_data_home,
        min_faces_per_person=70,
        resize=0.4
    )
else:
    # Version 2: no local copy (e.g. Google Colab), so download automatically.
    print("Local LFW cache not found; downloading with scikit-learn...")
    lfw_people = fetch_lfw_people(
        min_faces_per_person=70,
        resize=0.4
    )

X = lfw_people.images
y = lfw_people.target
target_names = lfw_people.target_names
n_classes = len(target_names)

print(f"LFW dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Number of classes: {n_classes}")
print(f"Pixel range: min={X.min():.4f}, max={X.max():.4f}")
print("Class names:", target_names)

## 4. Train–test splitThe split follows the lab specification exactly. No extra pixel normalisation is applied because the printed range is already approximately `[0, 1]`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print(f"Train images: {X_train.shape}, train labels: {y_train.shape}")
print(f"Test images:  {X_test.shape}, test labels:  {y_test.shape}")

## 5. Tensor conversion and DataLoadersPyTorch `Conv2d` expects `[N, C, H, W]`. LFW is grayscale, so a channel dimension of size 1 is inserted.

In [ ]:
X_train = X_train[:, np.newaxis, :, :]
X_test = X_test[:, np.newaxis, :, :]

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"CNN train tensor shape [N, C, H, W]: {X_train_tensor.shape}")
print(f"CNN test tensor shape  [N, C, H, W]: {X_test_tensor.shape}")

## 6. CNN classBoth required convolution layers use 32 filters and a 3×3 kernel. A dummy input computes the flattened feature size automatically, so the dense layer is not tied to a manually calculated image size.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes, image_height, image_width):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        # Infer the flatten size from the actual LFW image dimensions.
        with torch.no_grad():
            dummy = torch.zeros(1, 1, image_height, image_width)
            flattened_size = self.features(dummy).flatten(start_dim=1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

## 7. Model initialisation

In [ ]:
image_height, image_width = X_train.shape[2], X_train.shape[3]
model = SimpleCNN(n_classes, image_height, image_width).to(device)
print("Complete CNN architecture:")
print(model)

## 8. Loss and optimiser`CrossEntropyLoss` accepts integer class labels directly, so the labels must **not** be one-hot encoded.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("Loss: CrossEntropyLoss")
print("Optimizer: Adam (learning rate = 0.001)")

## 9. Training loopThe required train/test split is retained. The test accuracy shown after each epoch is recorded for the requested accuracy curve; the final evaluation below runs inference over the complete test set.

In [ ]:
def accuracy_on_loader(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total


NUM_EPOCHS = 15
train_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    epoch_loss = running_loss / train_total
    epoch_train_accuracy = train_correct / train_total
    epoch_test_accuracy = accuracy_on_loader(model, test_loader, device)

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_train_accuracy)
    test_accuracies.append(epoch_test_accuracy)

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Training Loss: {epoch_loss:.4f} | "
        f"Training Accuracy: {epoch_train_accuracy:.4f} | "
        f"Test Accuracy: {epoch_test_accuracy:.4f}"
    )

## 10. Evaluation on the complete test set

In [ ]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs.to(device))
        predictions = outputs.argmax(dim=1).cpu()
        all_predictions.extend(predictions.numpy())
        all_labels.extend(labels.numpy())

all_predictions = np.asarray(all_predictions)
all_labels = np.asarray(all_labels)
total_test_samples = len(all_labels)
total_correct = int((all_predictions == all_labels).sum())
cnn_accuracy = total_correct / total_test_samples

print(f"Total test samples: {total_test_samples}")
print(f"Total correct: {total_correct}")
print(f"CNN test accuracy: {cnn_accuracy:.4f} ({cnn_accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=np.arange(n_classes),
    target_names=target_names,
    zero_division=0
))

## 11. Training curves

In [ ]:
epochs = np.arange(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs, train_losses, marker="o", color="tab:blue")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, train_accuracies, marker="o", label="Training accuracy")
axes[1].plot(epochs, test_accuracies, marker="s", label="Test accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("part3_1_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Sample predictions

In [ ]:
rng = np.random.default_rng(SEED)
sample_count = min(8, len(X_test_tensor))
sample_indices = rng.choice(len(X_test_tensor), size=sample_count, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = np.asarray(axes).reshape(-1)

for ax, index in zip(axes, sample_indices):
    true_id = int(y_test_tensor[index])
    predicted_id = int(all_predictions[index])
    ax.imshow(X_test_tensor[index, 0].numpy(), cmap="gray")
    ax.set_title(
        f"Ground Truth: {target_names[true_id]}\n"
        f"Prediction: {target_names[predicted_id]}",
        color="green" if true_id == predicted_id else "red",
        fontsize=9,
    )
    ax.axis("off")

for ax in axes[sample_count:]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("part3_1_sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 2 comparison

In [ ]:
# Replace None with the Part 2 Random Forest accuracy, for example 0.75.
rf_accuracy = None

if rf_accuracy is None:
    print("Random Forest Accuracy: not entered (set rf_accuracy above)")
else:
    print(f"Random Forest Accuracy: {rf_accuracy:.4f} ({rf_accuracy * 100:.2f}%)")
print(f"CNN Accuracy:           {cnn_accuracy:.4f} ({cnn_accuracy * 100:.2f}%)")

# Demonstration explanation notes

### Conv2d
- `in_channels` is the number of channels entering the layer. The first layer receives one grayscale channel; the second receives the 32 feature maps produced by the first layer.
- `out_channels=32` means the layer learns 32 different filters and outputs 32 feature maps.
- `kernel_size=3` means each filter examines a local 3×3 pixel region at a time. During training, the filter weights are learned to detect useful patterns such as edges, textures, and facial parts.

### Why CNN input is 4D
PyTorch represents an image batch as `[batch, channel, height, width]`. Batch is the number of images processed together, channel is 1 for grayscale, and height and width preserve the 2D image layout.

### Why the input is not flattened first
A CNN uses local neighbourhoods: nearby pixels form meaningful structures such as eyes and edges. Flattening before convolution discards the explicit 2D arrangement, so a convolutional kernel could no longer move across local image regions. Flattening is only done after convolution and pooling have extracted spatial features.

### ReLU
ReLU computes `max(0, x)`. It introduces non-linearity, allowing stacked layers to learn more complex decision functions than one linear transformation.

### MaxPooling
A 2×2 max-pooling layer keeps the largest activation in each 2×2 region. This approximately halves feature-map height and width, reduces computation, and retains the strongest local responses.

### Dense layer
After the CNN has produced feature maps, `Flatten` converts them into one feature vector per image. The dense layers combine these learned features. The final dense layer outputs `n_classes` logits—one unnormalised score per person—and the largest score is the predicted class.

### CrossEntropyLoss
Cross-entropy is suitable for mutually exclusive multi-class classification. It compares the vector of class logits with the correct integer class ID, internally applying the required log-softmax calculation. Therefore, no manual softmax or one-hot labels are needed.

### Adam
Adam is the optimiser. It uses gradients from backpropagation to update the model parameters and adapts the effective step size for each parameter, helping minimise the training loss.

### CNN versus PCA / Eigenfaces + Random Forest
PCA is a fixed linear, unsupervised dimensionality-reduction method: it finds directions of high image variance without using class labels. The Random Forest then classifies those separately produced PCA features. A CNN instead learns spatial feature extraction and the classifier jointly, end-to-end, using the classification loss and labels. This does not guarantee that the CNN must achieve higher accuracy; the reported test results determine the comparison.